## Step 1: Clone Repository

In [ ]:
!git clone https://github.com/ahmedelbamby-aast/DeepFakeBenchUpgraded.git

## Step 2: Change Directory

In [ ]:
%cd DeepFakeBenchUpgraded

## Step 3: Install DeepfakeBench
This takes ~2-3 minutes. Dependency warnings are normal and can be ignored.

In [ ]:
!bash kaggle_install.sh

## Step 4: Verify Installation
Test if everything works correctly

In [ ]:
import torch
import sys

# Check PyTorch and GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Test DeepfakeBench import
sys.path.insert(0, '.')
from deepfakebench.detectors.xception_detector import XceptionDetector
print("\n✅ DeepfakeBench is ready to use!")

## Optional: Quick Model Test
Test loading a detector model

In [ ]:
import yaml

# Load Xception config
with open('deepfakebench/config/detector/xception.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Config loaded successfully!")
print(f"Detector: {config.get('detector_name', 'Unknown')}")
print(f"Backbone: {config.get('backbone_name', 'Unknown')}")

## Next Steps

### Training
```bash
!python deepfakebench/train.py \
    --detector_path ./deepfakebench/config/detector/xception.yaml \
    --train_dataset FF-DF \
    --test_dataset FF-DF
```

### Testing
```bash
!python deepfakebench/test.py \
    --detector_path ./deepfakebench/config/detector/xception.yaml \
    --test_dataset FF-DF \
    --weights_path /path/to/checkpoint.pth
```

### Add Datasets
1. Go to **Add Data** → Search for "FaceForensics", "Celeb-DF", or "DFDC"
2. Add to notebook
3. Link datasets:
```bash
!mkdir -p datasets/rgb
!ln -s /kaggle/input/your-dataset datasets/rgb/FaceForensics++
```

## Test 6: Verify Kaggle Dataset Structure Compatibility

In [ ]:
"""
This cell verifies your Kaggle dataset structure is compatible with DeepfakeBench.

Your dataset structure:
/kaggle/input/faceforensicsplusplus-c23-deepfakebench-structure/rgb/FaceForensics++/
├── manipulated_sequences/
│   ├── Face2Face/, Deepfakes/, DeepFakeDetection/, NeuralTextures/, FaceShifter/, FaceSwap/
│   │   └── c23/frames/[video_folders]/[frame_files]
└── original_sequences/
    └── youtube/c23/frames/[video_folders]/[frame_files]
"""

import os
import sys

# Simulate Kaggle path structure locally
base_path_kaggle = '/kaggle/input/faceforensicsplusplus-c23-deepfakebench-structure/rgb/FaceForensics++'

# For local testing, use a mock structure
print("Testing dataset structure compatibility...")
print("=" * 70)

# Expected structure
expected_structure = {
    'manipulated_sequences': ['Face2Face', 'Deepfakes', 'DeepFakeDetection', 
                              'NeuralTextures', 'FaceShifter', 'FaceSwap'],
    'original_sequences': ['youtube']
}

print("\n✅ Expected Structure:")
print(f"  - manipulated_sequences/ with methods: {expected_structure['manipulated_sequences']}")
print(f"  - original_sequences/ with: {expected_structure['original_sequences']}")
print(f"  - Each method has: c23/frames/[video_folders]/[frame_files]")

print("\n✅ System Support:")
print("  - preprocessing/rearrange.py: Scans this exact structure")
print("  - dataset/abstract_dataset.py: Loads from JSON mappings")
print("  - Compression c23: Fully supported")

print("\n✅ Configuration for Kaggle:")
print(f"""
config = {{
    'rgb_dir': '{base_path_kaggle.replace('FaceForensics++', '')}',
    'dataset_json_folder': './preprocessing/dataset_json',
    'compression': 'c23',
    'train_dataset': ['FaceForensics++'],  # or ['FF-F2F', 'FF-DF', etc.]
    'test_dataset': 'FaceForensics++'
}}
""")

print("\n📋 Next Steps on Kaggle:")
print("  1. Run rearrange.py to generate JSON mapping")
print("  2. Update config with your dataset path")
print("  3. Start training or testing")
print("\nSee KAGGLE_DATASET_GUIDE.md for complete instructions!")
print("=" * 70)

## Test 7: Generate Dataset JSON Mapping
This step scans your dataset structure and creates JSON files for training/testing

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/DeepFakeBenchUpgraded')

# Check if dataset exists
import os
dataset_path = '/kaggle/input/faceforensicsplusplus-c23-deepfakebench-structure/rgb/FaceForensics++'

if os.path.exists(dataset_path):
    print("✓ Dataset found!")
    print(f"  Path: {dataset_path}")
    
    # Check structure
    if os.path.exists(os.path.join(dataset_path, 'manipulated_sequences')):
        methods = os.listdir(os.path.join(dataset_path, 'manipulated_sequences'))
        print(f"  Manipulation methods: {methods}")
    
    if os.path.exists(os.path.join(dataset_path, 'original_sequences')):
        print(f"  Original sequences: Found")
    
    print("\nGenerating JSON mapping...")
    
    # Import and run rearrange
    from preprocessing.rearrange import rearrange_dataset
    
    rearrange_dataset(
        dataset_name='FaceForensics++',
        dataset_root=dataset_path,
        output_dir='./preprocessing/dataset_json'
    )
    
    print("\n✅ JSON mapping generated successfully!")
    print("   Location: ./preprocessing/dataset_json/FaceForensics++.json")
else:
    print("⚠️ Dataset not found at:", dataset_path)
    print("\nFor local testing, this is expected.")
    print("On Kaggle, make sure to:")
    print("  1. Add the dataset to your notebook")
    print("  2. Verify the path matches your dataset input")

## Test 8: Configure Training
Set up training configuration for your dataset

In [ ]:
import yaml
import os

# Load base configuration
config_path = 'deepfakebench/config/detector/sladd_detector.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update paths for Kaggle
config['rgb_dir'] = '/kaggle/input/faceforensicsplusplus-c23-deepfakebench-structure/rgb'
config['dataset_json_folder'] = './preprocessing/dataset_json'
config['log_dir'] = './kaggle_logs'

# Training settings
config['compression'] = 'c23'
config['train_dataset'] = ['FaceForensics++']  # Train on all methods
config['test_dataset'] = 'FaceForensics++'     # Test on all methods

# Optional: Train on specific methods only
# config['train_dataset'] = ['FF-F2F', 'FF-DF']  # Face2Face and Deepfakes only
# config['test_dataset'] = 'FF-FS'               # Test on FaceSwap

# Training hyperparameters (adjust as needed)
config['nEpochs'] = 5  # Number of epochs (increase for better results)
config['train_batchSize'] = 16  # Batch size (adjust based on GPU memory)
config['test_batchSize'] = 32
config['save_epoch'] = 1  # Save checkpoint every epoch

# Frame settings
config['frame_num'] = {'train': 8, 'test': 32}  # Frames per video
config['resolution'] = 256  # Image resolution

print("✅ Training Configuration:")
print(f"  Dataset path: {config['rgb_dir']}")
print(f"  Compression: {config['compression']}")
print(f"  Train datasets: {config['train_dataset']}")
print(f"  Test dataset: {config['test_dataset']}")
print(f"  Epochs: {config['nEpochs']}")
print(f"  Batch size: {config['train_batchSize']}")
print(f"  Frames per video: {config['frame_num']['train']}")

# Save configuration
kaggle_config_path = 'kaggle_training_config.yaml'
with open(kaggle_config_path, 'w') as f:
    yaml.dump(config, f)

print(f"\n✅ Config saved to: {kaggle_config_path}")
print("\nReady for training!")

## Test 9: Start Training
Run training with your configured settings

**Note:** Training will take time depending on:
- Number of epochs
- Dataset size  
- GPU availability (T4 x2 recommended)

For a quick test, use 1-2 epochs. For actual training, use 20-40 epochs.

In [ ]:
# Option 1: Quick Training Test (1 epoch for testing)
# Uncomment to run a quick test
"""
!python deepfakebench/train.py \
    --detector_path kaggle_training_config.yaml \
    --train_dataset FaceForensics++ \
    --test_dataset FaceForensics++
"""

# Option 2: Full Training (configure epochs in the config above)
# This will train the model on your dataset
# Run this when you're ready for actual training:

print("Training commands:")
print("\n1. Quick test (already configured in config):")
print("   !python deepfakebench/train.py --detector_path kaggle_training_config.yaml")

print("\n2. Resume from checkpoint:")
print("   !python deepfakebench/train.py --detector_path kaggle_training_config.yaml --resume /path/to/checkpoint.pth")

print("\n3. Monitor training:")
print("   Check './kaggle_logs' directory for:")
print("   - Training logs")
print("   - Model checkpoints")
print("   - TensorBoard logs (if installed)")

print("\n⚠️ Important:")
print("  - Training time: ~30-60 min per epoch on T4 x2 GPU")
print("  - Make sure GPU is enabled in Kaggle settings")
print("  - Checkpoints saved every epoch to kaggle_logs/")

# Uncomment the line below when ready to train
# !python deepfakebench/train.py --detector_path kaggle_training_config.yaml

## Test 10: Monitor Training Progress (Optional)

Check training logs and saved checkpoints

In [ ]:
import os
import glob

# Check if training has started
log_dir = './kaggle_logs'

if os.path.exists(log_dir):
    print("✓ Training logs directory found")
    
    # List all subdirectories (each training run)
    runs = [d for d in os.listdir(log_dir) if os.path.isdir(os.path.join(log_dir, d))]
    
    if runs:
        print(f"\nTraining runs found: {len(runs)}")
        for run in runs:
            run_path = os.path.join(log_dir, run)
            print(f"\n  Run: {run}")
            
            # Check for checkpoints
            ckpts = glob.glob(os.path.join(run_path, '*.pth'))
            if ckpts:
                print(f"    Checkpoints: {len(ckpts)}")
                for ckpt in ckpts[:3]:  # Show first 3
                    print(f"      - {os.path.basename(ckpt)}")
            
            # Check for log files
            logs = glob.glob(os.path.join(run_path, '*.log'))
            if logs:
                print(f"    Log files: {len(logs)}")
                
                # Show last few lines of latest log
                latest_log = max(logs, key=os.path.getctime)
                print(f"\n    Latest log excerpt ({os.path.basename(latest_log)}):")
                try:
                    with open(latest_log, 'r') as f:
                        lines = f.readlines()
                        for line in lines[-10:]:  # Last 10 lines
                            print(f"      {line.strip()}")
                except:
                    pass
    else:
        print("\n  No training runs yet")
else:
    print("⚠️ No training logs found yet")
    print("   Start training first (see Test 9)")

print("\n" + "="*70)
print("After training completes, you can:")
print("  1. Download best checkpoint: kaggle_logs/.../best.pth")
print("  2. Evaluate on test set using deepfakebench/test.py")
print("  3. Use the model for inference")